In [1]:
from tensorflow.keras.preprocessing.text import one_hot

In [3]:
### sentences
paragraph=[  'the glass of milk',
     'the glass of juice',
     'the cup of tea',
    'I am a good boy',
     'I am a good developer',
     'understand the meaning of words',
     'your videos are good']

## Define the vocabulary size
voc_size=10000

In [ ]:
one_hot_repr = [ one_hot(sent,voc_size) for sent in paragraph]
print(one_hot_repr)

'''
[8151, 3226, 6001, 4870]
8151 in the index of 'the' word , it means at 8151 there will be 1 rest all are zeros
3226 in the index of 'glass' word , it means at 3226 there will be 1 rest all are zeros
'''

[[8151, 3226, 6001, 4870], [8151, 3226, 6001, 7744], [8151, 7809, 6001, 3348], [7175, 8812, 1390, 6063, 5463], [7175, 8812, 1390, 6063, 8331], [2009, 8151, 4768, 6001, 8458], [4428, 3792, 9991, 6063]]


In [10]:
## word Embedding Representation

from keras.layers import Embedding
from keras.preprocessing.sequence import pad_sequences
from keras.models import Sequential
from keras import Input
from keras.optimizers import Adam
from keras.losses import MeanAbsoluteError
import numpy as np


In [7]:
sent_length=8
embedded_docs = pad_sequences(sequences=one_hot_repr,padding='pre',maxlen=sent_length)
print(embedded_docs)

[[   0    0    0    0 8151 3226 6001 4870]
 [   0    0    0    0 8151 3226 6001 7744]
 [   0    0    0    0 8151 7809 6001 3348]
 [   0    0    0 7175 8812 1390 6063 5463]
 [   0    0    0 7175 8812 1390 6063 8331]
 [   0    0    0 2009 8151 4768 6001 8458]
 [   0    0    0    0 4428 3792 9991 6063]]


The problem: a neural network wants every input to be the same length, but your sentences aren't.

After one_hot, each sentence is a list of numbers, one number per word:

"the glass of milk"          → [6523, 4122, 7541, 9876]           (4 words)
"I am a good boy"            → [1234, 5543, 222, 8765, 4410]      (5 words)
"understand the meaning"     → [3355, 6523, 9001]                 (3 words)

The lists come out 4, 5 and 3 long, and the model can't handle different lengths.

The fix: pad_sequences makes every list exactly the same length by filling in zeros.

- maxlen=sent_length (8): make every sentence exactly 8 numbers long.
- padding='pre': put the zeros at the front. ('post' would put them at the end.)

The result:

[   0    0    0    0 6523 4122 7541 9876]
[   0    0    0 1234 5543  222 8765 4410]
[   0    0    0    0    0 3355 6523 9001]

Every row is now 8 numbers long, and the real words are still there, just pushed to the right.

Why zeros? one_hot never gives a word the number 0, so 0 is free to mean "empty spot, no word here."

In [11]:
## feature representation

dim = 10
opt = Adam(learning_rate=0.01)
loss = MeanAbsoluteError()

model = Sequential()
model.add(Input(shape=(sent_length,)))
model.add(Embedding(input_dim=voc_size,output_dim=dim))
model.compile(optimizer=opt,loss=loss)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 8, 10)          │       100,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 100,000 (390.62 KB)

 Trainable params: 100,000 (390.62 KB)

 Non-trainable params: 0 (0.00 B)

The big picture: you're building a machine that turns each word number into a short list of 10 numbers that describes the word.

model.add(Input(shape=(sent_length,)))
Put a label on the box's entrance: "Every sentence that comes in has exactly 8 numbers."
[   0    0    0    0 8151 3226 6001 4870]

model.add(Embedding(input_dim=voc_size, output_dim=dim))
- It has 10,000 rows (voc_size), one row for every possible word number.
- Each row holds 10 numbers (dim), which describe that word.

row 0     → [ 0.01, -0.03,  0.04, ... ]   (10 numbers)
row 1     → [ 0.02,  0.00, -0.01, ... ]
...
row 8151  → [-0.04,  0.02,  0.03, ... ]   ← "the"
...
row 9999  → [ ... ]

When the sentence [0 0 0 0 8151 3226 6001 4870] comes in, the layer looks up each number's row:

- 0 → look for row 0 → 10 numbers
- 8151 → look for row 8151 → 10 numbers
- … and so on for all 8 words.

Input: 8 numbers → Output: 8 words × 10 numbers each = shape (8, 10).

example
INPUT  (after pad_sequences, sent_length = 8)
[   0,    0,    0,    0, 8151, 3226, 6001, 4870 ]
  pad   pad   pad   pad  'the' 'glass' 'of' 'milk'

EMBEDDING LAYER  (voc_size = 10000 rows, dim = 10 columns)
Each number → look up that row → get 10 numbers

OUTPUT  shape = (8, 10)   → 8 words, 10 numbers per word

               d1     d2     d3     d4     d5     d6     d7     d8     d9     d10
pos1 (0)    [ 0.02, -0.01,  0.04,  0.03, -0.02,  0.01,  0.00, -0.04,  0.02,  0.03 ]  ← row 0
pos2 (0)    [ 0.02, -0.01,  0.04,  0.03, -0.02,  0.01,  0.00, -0.04,  0.02,  0.03 ]  ← row 0
pos3 (0)    [ 0.02, -0.01,  0.04,  0.03, -0.02,  0.01,  0.00, -0.04,  0.02,  0.03 ]  ← row 0
pos4 (0)    [ 0.02, -0.01,  0.04,  0.03, -0.02,  0.01,  0.00, -0.04,  0.02,  0.03 ]  ← row 0
pos5 (8151) [-0.04,  0.02,  0.03, -0.01,  0.04,  0.00, -0.03,  0.01,  0.02, -0.02 ]  ← 'the'
pos6 (3226) [ 0.01,  0.04, -0.02,  0.03,  0.00, -0.04,  0.02,  0.03, -0.01,  0.04 ]  ← 'glass'
pos7 (6001) [-0.03,  0.00,  0.01, -0.04,  0.02,  0.03, -0.01,  0.04,  0.00, -0.02 ]  ← 'of'
pos8 (4870) [ 0.03, -0.02, -0.04,  0.01,  0.03,  0.02,  0.04, -0.03,  0.01,  0.00 ]  ← 'milk'


Right now the table is filled with small random numbers. Only after training would words with similar meanings ("good" and "great") end up with similar rows.

Line 4: model.compile(optimizer=opt, loss=loss)
Get the machine ready to learn:
- loss (MeanAbsoluteError) is the scorecard: how wrong is the machine?
- optimizer (Adam) is the mechanic: it adjusts the table's numbers to lower that score.

In this demo you never actually train, so compile just gets everything ready. You only call predict to see the vectors.



In [13]:
# predict the values for 8 words length
one_hot_embeded_word_number = np.array([[   0,    0,    0, 7175, 8812, 1390, 6063, 8331]])
predicted_value = model.predict(one_hot_embeded_word_number)
print(predicted_value)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
[[[-5.7092421e-03 -3.9169692e-02  5.0639138e-03 -5.1520765e-05
    3.1172562e-02  2.2169176e-02 -3.3917867e-02 -1.5176311e-03
    2.8704118e-02  4.4419531e-02]
  [-5.7092421e-03 -3.9169692e-02  5.0639138e-03 -5.1520765e-05
    3.1172562e-02  2.2169176e-02 -3.3917867e-02 -1.5176311e-03
    2.8704118e-02  4.4419531e-02]
  [-5.7092421e-03 -3.9169692e-02  5.0639138e-03 -5.1520765e-05
    3.1172562e-02  2.2169176e-02 -3.3917867e-02 -1.5176311e-03
    2.8704118e-02  4.4419531e-02]
  [ 3.2036413e-02  4.3130998e-02 -3.0331170e-02  3.5611000e-02
    1.4361534e-02  2.7112458e-02  2.0460580e-02 -8.6721182e-03
    7.9945549e-03  1.4995445e-02]
  [-2.3926688e-02 -3.4329273e-02  9.5882267e-04  4.3520216e-02
   -2.6565457e-02 -2.0170940e-02  4.2244196e-03  3.5310213e-02
    8.0950744e-03 -1.2520254e-02]
  [-2.3196364e-02  9.2615969e-03  1.2202360e-02  7.4805506e-03
    1.9040551e-02 -2.0669555e-02 -2.6110459e-02 -4.7326174e-02
    2.7590130e-02 -3.9902925e-03]
  